In [1]:
import numpy as np
import pandas as pd
import os
import copy
import glob
import random
from random import random as rd
import gc

df_train = pd.read_csv("../input/ventilator-pressure-prediction/train.csv")

unique_pressures = df_train["pressure"].unique()
sorted_pressures = np.sort(unique_pressures)
total_pressures_len = len(sorted_pressures)
min_pressure = float(sorted_pressures[0])
max_pressure = float(sorted_pressures[-1])


def find_nearest(prediction):
    insert_idx = np.searchsorted(sorted_pressures, prediction)
    if insert_idx == total_pressures_len:
        return sorted_pressures[-1]
    elif insert_idx == 0:
        return sorted_pressures[0]
    lower_val = sorted_pressures[insert_idx - 1]
    upper_val = sorted_pressures[insert_idx]
    return (
        lower_val
        if abs(lower_val - prediction) < abs(upper_val - prediction)
        else upper_val
    )


def snap_to_known_pressures(pred_arr: np.ndarray) -> np.ndarray:
    pred_arr = pred_arr.astype(np.float64, copy=False)
    idx = np.searchsorted(sorted_pressures, pred_arr, side="left")
    idx = np.clip(idx, 0, total_pressures_len - 1)

    lower_idx = np.clip(idx - 1, 0, total_pressures_len - 1)
    upper_idx = idx

    lower = sorted_pressures[lower_idx]
    upper = sorted_pressures[upper_idx]

    choose_lower = np.abs(pred_arr - lower) < np.abs(pred_arr - upper)
    out = np.where(choose_lower, lower, upper)
    return out.astype(np.float32, copy=False)


def set_seed(seed=2021):
    np.random.seed(seed)
    random_state = np.random.RandomState(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    return random_state


def wc(input_list):
    l = []
    for i in range(len(input_list)):
        public_lb_score = int(input_list[i].split("/")[-1].split(".")[1].split(" ")[0])
        l.append(public_lb_score)
        input_list[i] = (pd.read_csv(input_list[i]).pressure).ravel()
    output = 0
    l_sum = sum(l)
    if len(input_list) == 1:
        output = input_list[0]
    else:
        weight1 = (l[1] / l_sum) + 0.15
        weight2 = 1 - weight1
        output += input_list[0] * weight1 + input_list[1] * weight2
    return output


def g(dp):
    l = []
    for i in glob.iglob(f"{dp}/*"):
        l.append(i)
    file_count = len(l)
    loop_time = 154
    splits = file_count // 2
    l.sort()
    flist = []
    for i in range(splits):
        if i == splits - 1:
            flist.append(l[i * round(len(l) / splits) :])
        else:
            flist.append(
                l[i * round(len(l) / splits) : (i + 1) * round(len(l) / splits)]
            )
    for i in range(len(flist)):
        flist[i] = wc(flist[i])
    pred_list = []
    for i in range(loop_time):
        weight = []
        set_seed(i)
        for i in range(len(flist)):
            weight.append(rd())
        weight_sum = sum(weight)
        for i in range(len(weight)):
            weight[i] /= weight_sum
        weight.sort(reverse=True)
        temp = 0
        for i in range(len(flist)):
            temp += flist[i] * weight[i]
        pred_list.append(temp)
        del temp
        gc.collect()
    output = pd.read_csv(
        "../input/ventilator-pressure-prediction/sample_submission.csv"
    )
    output.pressure = np.median(np.vstack(pred_list), axis=0)
    output["pressure"] = output["pressure"].apply(find_nearest)
    output.to_csv(f"rwb {loop_time} loops.csv", index=False)


def blend(a, b):
    a = pd.read_csv(a)
    b = pd.read_csv(b)
    a.pressure = a.pressure * 0.5 + b.pressure * 0.5
    a["pressure"] = a["pressure"].apply(find_nearest)
    a.to_csv("blend.csv", index=False)
    return a




In [2]:
df_test = pd.read_csv("../input/ventilator-pressure-prediction/test.csv")


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["breath_id", "time_step"]).copy()
    g = df.groupby("breath_id", sort=False)

    df["u_in_lag1"] = g["u_in"].shift(1).fillna(0.0)
    df["u_in_lag2"] = g["u_in"].shift(2).fillna(0.0)
    df["u_out_lag1"] = g["u_out"].shift(1).fillna(0).astype(np.int64)

    dt = g["time_step"].diff().fillna(0.0)
    df["dt"] = dt

    df["du_in"] = (df["u_in"] - df["u_in_lag1"]).astype(np.float64)
    df["du_in_dt"] = np.where(df["dt"].to_numpy() > 0, df["du_in"] / df["dt"], 0.0)

    df["u_in_cum"] = g["u_in"].cumsum()
    df["u_in_area"] = (df["u_in"] * dt).groupby(df["breath_id"], sort=False).cumsum()

    eff_u_in = df["u_in"] * (1.0 - df["u_out"].astype(np.float64))
    df["eff_u_in_area"] = (eff_u_in * dt).groupby(df["breath_id"], sort=False).cumsum()

    df["t2"] = df["time_step"].astype(np.float64) ** 2

    df["breath_step"] = g.cumcount().astype(np.int16)  # 0..79
    df["breath_step2"] = (df["breath_step"].astype(np.float64) ** 2).astype(np.float64)
    df["breath_step3"] = (df["breath_step"].astype(np.float64) ** 3).astype(np.float64)

    df["u_in_x_time"] = (
        df["u_in"].astype(np.float64) * df["time_step"].astype(np.float64)
    ).astype(np.float64)
    df["u_in_x_step"] = (
        df["u_in"].astype(np.float64) * df["breath_step"].astype(np.float64)
    ).astype(np.float64)

    if "pressure" in df.columns:
        df["pressure_lag1"] = g["pressure"].shift(1).fillna(0.0).astype(np.float64)
    else:
        df["pressure_lag1"] = 0.0

    return df


train_fe = add_time_features(df_train)
test_fe = add_time_features(df_test)

train_insp = train_fe[train_fe["u_out"] == 0].copy()
global_insp_mean = float(train_insp["pressure"].mean())



In [3]:
FEATURES = [
    "time_step",
    "t2",
    "breath_step",
    "breath_step2",
    "breath_step3",
    "u_in",
    "u_in_lag1",
    "u_in_lag2",
    "du_in",
    "du_in_dt",
    "u_in_cum",
    "u_in_area",
    "eff_u_in_area",
    "u_in_x_time",
    "u_in_x_step",
    "u_out",
    "u_out_lag1",
    "pressure_lag1",
]

feat_mu = train_insp[FEATURES].astype(np.float64).mean(axis=0).to_numpy()
feat_sigma = (
    train_insp[FEATURES].astype(np.float64).std(axis=0).replace(0.0, 1.0).to_numpy()
)


def _build_design(df: pd.DataFrame) -> np.ndarray:
    X = df[FEATURES].astype(np.float64).to_numpy()
    X = (X - feat_mu) / feat_sigma
    ones = np.ones((X.shape[0], 1), dtype=np.float64)
    return np.hstack([ones, X])


def fit_ridge_closed_form(
    X: np.ndarray,
    y: np.ndarray,
    alpha: float = 1e-3,
    sample_weight: np.ndarray | None = None,
) -> np.ndarray:
    if sample_weight is None:
        XtX = X.T @ X
        Xty = X.T @ y
    else:
        w = sample_weight.astype(np.float64, copy=False)
        Xw = X * w[:, None]
        XtX = X.T @ Xw
        Xty = X.T @ (y * w)

    reg = np.eye(XtX.shape[0], dtype=np.float64) * alpha
    reg[0, 0] = 0.0
    coef = np.linalg.solve(XtX + reg, Xty)
    return coef


step_norm = train_insp["breath_step"].astype(np.float64).to_numpy() / 79.0
global_sw = 0.7 + 0.6 * step_norm  # range ~[0.7, 1.3]

weights_pass1 = {}
Xg = _build_design(train_insp)
yg = train_insp["pressure"].astype(np.float64).to_numpy()
fallback_w1 = fit_ridge_closed_form(Xg, yg, alpha=1e-2, sample_weight=global_sw)

for (R, C), df_rc in train_insp.groupby(["R", "C"], sort=False):
    X = _build_design(df_rc)
    y = df_rc["pressure"].astype(np.float64).to_numpy()
    step_norm_rc = df_rc["breath_step"].astype(np.float64).to_numpy() / 79.0
    sw_rc = 0.7 + 0.6 * step_norm_rc
    w = fit_ridge_closed_form(X, y, alpha=5e-2, sample_weight=sw_rc)
    weights_pass1[(int(R), int(C))] = w

p_feat_idx = FEATURES.index("pressure_lag1")
p_mu = float(feat_mu[p_feat_idx])
p_sig = float(feat_sigma[p_feat_idx])

train_insp2 = train_insp.sort_values(["breath_id", "time_step"]).copy()

pred_train = np.empty(len(train_insp2), dtype=np.float64)

X_feat_tr = train_insp2[FEATURES].astype(np.float64).to_numpy()
X_norm_tr = (X_feat_tr - feat_mu) / feat_sigma
X_base_tr = np.hstack([np.ones((X_norm_tr.shape[0], 1), dtype=np.float64), X_norm_tr])

base_pnorm_tr = X_base_tr[:, 1 + p_feat_idx]
breath_ids_tr = train_insp2["breath_id"].to_numpy()
starts_tr = np.r_[0, np.flatnonzero(breath_ids_tr[1:] != breath_ids_tr[:-1]) + 1]
ends_tr = np.r_[starts_tr[1:], len(breath_ids_tr)]
R_tr = train_insp2["R"].to_numpy(dtype=np.int16, copy=False)
C_tr = train_insp2["C"].to_numpy(dtype=np.int16, copy=False)

for s, e in zip(starts_tr, ends_tr):
    Rv = int(R_tr[s])
    Cv = int(C_tr[s])
    w1 = weights_pass1.get((Rv, Cv), fallback_w1)

    Xb = X_base_tr[s:e]
    base_pred = Xb @ w1
    w_p = float(w1[1 + p_feat_idx])

    prev_p = global_insp_mean
    for t in range(e - s):
        if t == 0:
            p_lag = 0.0
        else:
            p_lag = float(
                snap_to_known_pressures(np.array([prev_p], dtype=np.float64))[0]
            )
        p_norm = (p_lag - p_mu) / p_sig
        pj = float(base_pred[t] + w_p * (p_norm - base_pnorm_tr[s + t]))
        if not np.isfinite(pj):
            pj = global_insp_mean
        pj = float(np.clip(pj, min_pressure, max_pressure))
        pred_train[s + t] = pj
        prev_p = pj

pred_train_snap = snap_to_known_pressures(
    np.clip(pred_train, min_pressure, max_pressure)
).astype(np.float64)
train_insp2["pressure_lag1"] = (
    pd.Series(pred_train_snap, index=train_insp2.index)
    .groupby(train_insp2["breath_id"], sort=False)
    .shift(1)
    .fillna(0.0)
    .astype(np.float64)
)

feat_mu2 = train_insp2[FEATURES].astype(np.float64).mean(axis=0).to_numpy()
feat_sigma2 = (
    train_insp2[FEATURES].astype(np.float64).std(axis=0).replace(0.0, 1.0).to_numpy()
)

feat_mu = feat_mu2
feat_sigma = feat_sigma2


def _build_design(df: pd.DataFrame) -> np.ndarray:
    X = df[FEATURES].astype(np.float64).to_numpy()
    X = (X - feat_mu) / feat_sigma
    ones = np.ones((X.shape[0], 1), dtype=np.float64)
    return np.hstack([ones, X])


step_norm2 = train_insp2["breath_step"].astype(np.float64).to_numpy() / 79.0
global_sw2 = 0.7 + 0.6 * step_norm2

Xg2 = _build_design(train_insp2)
yg2 = train_insp2["pressure"].astype(np.float64).to_numpy()
fallback_w = fit_ridge_closed_form(Xg2, yg2, alpha=1e-2, sample_weight=global_sw2)

weights = {}
for (R, C), df_rc in train_insp2.groupby(["R", "C"], sort=False):
    X = _build_design(df_rc)
    y = df_rc["pressure"].astype(np.float64).to_numpy()
    step_norm_rc = df_rc["breath_step"].astype(np.float64).to_numpy() / 79.0
    sw_rc = 0.7 + 0.6 * step_norm_rc
    w = fit_ridge_closed_form(X, y, alpha=5e-2, sample_weight=sw_rc)
    weights[(int(R), int(C))] = w

p_feat_idx = FEATURES.index("pressure_lag1")
pcol = 1 + p_feat_idx
p_mu = float(feat_mu[p_feat_idx])
p_sig = float(feat_sigma[p_feat_idx])



In [4]:
test_fe_sorted = test_fe.sort_values(["breath_id", "time_step"]).reset_index(drop=False)

# Change (alignment fix): keep both the original row positions and the corresponding ids,
# then write the submission using the ids mapped by the same inverse-index used for predictions.
# This prevents any silent mismatch between df_test order and restored prediction order.
orig_pos = test_fe_sorted[
    "index"
].to_numpy()  # original df_test row position for each sorted row
ids_sorted = test_fe_sorted["id"].to_numpy(dtype=np.int64, copy=False)

X_feat = test_fe_sorted[FEATURES].astype(np.float64).to_numpy()
X_norm = (X_feat - feat_mu) / feat_sigma
X_base = np.hstack([np.ones((X_norm.shape[0], 1), dtype=np.float64), X_norm])

breath_ids = test_fe_sorted["breath_id"].to_numpy()
starts = np.r_[0, np.flatnonzero(breath_ids[1:] != breath_ids[:-1]) + 1]
ends = np.r_[starts[1:], len(breath_ids)]

R_arr = test_fe_sorted["R"].to_numpy(dtype=np.int16, copy=False)
C_arr = test_fe_sorted["C"].to_numpy(dtype=np.int16, copy=False)
u_out_sorted = test_fe_sorted["u_out"].to_numpy(dtype=np.int8, copy=False)

pred_sorted = np.empty(len(test_fe_sorted), dtype=np.float64)

for s, e in zip(starts, ends):
    Rv = int(R_arr[s])
    Cv = int(C_arr[s])
    w = weights.get((Rv, Cv), fallback_w)

    Xb = X_base[s:e]
    base_pred = Xb @ w
    base_pnorm = Xb[:, pcol]
    w_p = float(w[pcol])

    prev_p = global_insp_mean
    last_insp_p = global_insp_mean

    for t in range(e - s):
        if u_out_sorted[s + t] == 0:
            if t == 0:
                p_lag = 0.0
            else:
                p_lag = float(
                    snap_to_known_pressures(np.array([prev_p], dtype=np.float64))[0]
                )

            p_norm = (p_lag - p_mu) / p_sig
            pj = float(base_pred[t] + w_p * (p_norm - base_pnorm[t]))
            if not np.isfinite(pj):
                pj = global_insp_mean
            pj = float(np.clip(pj, min_pressure, max_pressure))

            pred_sorted[s + t] = pj
            prev_p = pj
            last_insp_p = pj
        else:
            pred_sorted[s + t] = last_insp_p

pred = np.empty(len(df_test), dtype=np.float64)
pred[orig_pos] = pred_sorted

# Sanity check: predicted array must align with original test rows 1:1
if pred.shape[0] != df_test.shape[0]:
    raise RuntimeError(f"pred rows {pred.shape[0]} != df_test rows {df_test.shape[0]}")

pred = np.where(np.isfinite(pred), pred, global_insp_mean)
pred = np.clip(pred, min_pressure, max_pressure)
pred_snap = snap_to_known_pressures(pred)

# Change (alignment fix): build ids in the same restored order used for pred[]
ids = np.empty(len(df_test), dtype=np.int64)
ids[orig_pos] = ids_sorted

submission = pd.DataFrame({"id": ids, "pressure": pred_snap})
submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)
print(
    "Pred stats (pre-snap):",
    float(np.min(pred)),
    float(np.mean(pred)),
    float(np.max(pred)),
)
print(
    "Pred stats (snapped):",
    float(np.min(pred_snap)),
    float(np.mean(pred_snap)),
    float(np.max(pred_snap)),
)
print("Inspiratory mean (train):", global_insp_mean)
print("Fitted per-(R,C) models:", len(weights))
print("Pressure range:", min_pressure, max_pressure)

   id   pressure
0   1   3.939334
1   2   6.048398
2   3   7.454441
3   4   8.930786
4   5  10.828944
Wrote submission.csv with shape: (603600, 2)
Pred stats (pre-snap): -1.895744295 19.83329848982126 64.82099174
Pred stats (snapped): -1.8957443237304688 19.832979202270508 64.82099151611328
Inspiratory mean (train): 17.589733414822938
Fitted per-(R,C) models: 9
Pressure range: -1.895744295 64.82099174
